# PornWorks Real Porn V03 SDXL — Google Colab

A separate notebook for the Diffusers-format checkpoint `John6666/pornworks-real-porn-v03-sdxl`. The Hugging Face repository is about 6.94 GB and uses the SDXL pipeline; first load will download several gigabytes. A T4 GPU may work with CPU offload, but generation can be slow and GPU availability is not guaranteed.

Use only with fictional adults (18+) and lawful, consensual content. Never create sexual imagery of minors or non-consensual intimate imagery of real people. Review the model's license and card before use: https://huggingface.co/John6666/pornworks-real-porn-v03-sdxl

In [ ]:
# Run this setup cell first.
%pip uninstall -y torchao
import subprocess, sys
cmd = [sys.executable, '-m', 'pip', 'install', '--upgrade', '--only-binary=:all:', 'diffusers', 'transformers>=4.48.0', 'tokenizers>=0.21.0', 'accelerate', 'safetensors']
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError('Package installation failed. Do not run later cells. Disconnect and delete the Colab runtime, reconnect, and retry this setup cell.')
print('Install succeeded. Restart the session, then run cells from the top.')

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU detected. In Colab choose Runtime → Change runtime type → GPU.')
props = torch.cuda.get_device_properties(0)
print(f'GPU: {props.name}')
print(f'VRAM: {props.total_memory / 1024**3:.1f} GiB')
print(f'PyTorch: {torch.__version__}')

## Load PornWorks V03

The model card documents this repository as a Diffusers SDXL pipeline. This cell uses FP16 for typical Colab T4 GPUs and CPU offload to reduce VRAM pressure.

In [ ]:
import torch
from diffusers import DiffusionPipeline

MODEL_ID = 'John6666/pornworks-real-porn-v03-sdxl'
pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print('PornWorks V03 loaded.')

## Generate an image

Edit the prompt below. Start at 768×768 on a T4 if memory or speed is a concern; 1024×1024 may be slower. Use fictional adult subjects only.

In [ ]:
from pathlib import Path
from IPython.display import display

prompt = 'Photorealistic editorial portrait of a fictional adult woman, age 28, elegant styling, natural skin texture, realistic eyes, soft studio lighting, 85mm lens, shallow depth of field, tasteful composition, high detail'
negative_prompt = 'low quality, blurry, plastic skin, malformed hands, extra fingers, distorted anatomy, text, watermark, logo, jpeg artifacts'
width = 768
height = 768
steps = 28
guidance = 5.0
seed = 123456

if width % 8 or height % 8:
    raise ValueError('Width and height must be multiples of 8.')
generator = torch.Generator(device='cuda').manual_seed(seed)
with torch.inference_mode():
    result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=width,
        height=height,
        num_inference_steps=steps,
        guidance_scale=guidance,
        num_images_per_prompt=1,
        generator=generator,
    )
image = result.images[0]
display(image)
output_path = Path('/content/pornworks_v03_output.png')
image.save(output_path)
print(f'Saved image to: {output_path}')

In [ ]:
from google.colab import files
files.download(str(output_path))

## Troubleshooting
- **Old code still appears:** Close the old tab and reopen this new notebook.
- **CUDA out of memory:** Restart session, use 640×640 or 512×512, and rerun the model-loading and generation cells.
- **Model download is slow:** The checkpoint is several gigabytes; wait for the first download to finish.
- **Setup fails:** Do not run later cells; share the complete error output.
- Review the model card and license before distributing outputs.